### 01. 데이터 수집·원본 검증 (P1)

계약된 원본이 다 모였는지(H00), 구조·품질 계약을 통과하는지(H01) 확인한다.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.pipeline.graph import Graph
from src.pipeline import manifest as mf
from src.pipeline.report import node_metrics, node_table

GRAPH = Graph.load()

### P1 노드 상태

In [2]:
node_table(*GRAPH.select(phase='P1'))

,이름,게이트,상태,검증,승인,실패 시,node_id
0,데이터 접근신청 현황,H00,pass,code / human,R1,h00_access_requests,h00_access_requests
1,강수량 수집 확인,H00,pass,code / human,-,h00_collect_rainfall,h00_collect_rainfall
2,하천수위 수집 확인,H00,pass,code / human,-,h00_collect_river,h00_collect_river
3,펌프장·하천 목록 수집 확인,H00,pass,code / human,-,h00_collect_small_tables,h00_collect_small_tables
4,SGIS 인구 통계·경계 수집 확인,H00,pass,code / human,-,h00_collect_sgis,h00_collect_sgis
5,토지피복·DEM 수집 확인,H00,pass,code / human,-,h00_collect_geo,h00_collect_geo
6,강수량 원본 검증,H01,pass,code / code / human,R1,h00_collect_rainfall,h01_contract_rainfall
7,하천수위 원본 검증,H01,pass,code / code / human,R1,h00_collect_river,h01_contract_river
8,펌프장·하천 목록 원본 검증,H01,pass,code / code / human,R1,h00_collect_small_tables,h01_contract_small_tables
9,SGIS 인구 통계·경계 원본 검증,H01,pass,code / code / human,R1,h00_collect_sgis,h01_contract_sgis


### 접근신청 현황

★ 표시 3건만 현재 분석에 필수다. 나머지 14건은 보조 자료이거나 뒤 단계(H09 알림)용이라
미신청이어도 P1~P3 진행에 지장이 없다.

In [3]:
rows = json.loads((ROOT / "artifacts/validation/access_requests.json").read_text(encoding="utf-8"))["rows"]
df = pd.DataFrame(rows)
df["필수"] = df["required"].map({True: "★", False: ""})
df[["no", "필수", "target", "status", "requested_at"]]

,no,필수,target,status,requested_at
0,1,★,하수도 민원 3년 + 관로 연도·좌표 + 침수흔적도 SHP,대기,2026-08-19
1,2,★,토지피복도(2025 중분류) 창원시,수령,2026-08-19 수령 확인
2,3,★,공개 DEM 창원시 도엽,수령,2026-08-19 수령 확인
3,4,,SGIS OpenAPI 인증키,미신청,NaN
4,5,,건축HUB 건축물대장 API 키,미신청,NaN
5,6,,기상청 API 키 (ASOS/특보),미신청,NaN
6,7,,강수·수위 관측지점 좌표 코드북,미신청,NaN
7,8,,재난안전데이터공유플랫폼 회원가입 + API 이용신청 3건: 침수흔적도 DSSP-IF...,미신청,NaN
8,9,,기상청 **API허브** 인증키 (AWS 매분·지점정보) — #6 data.go.k...,미신청,NaN
9,10,,V-World 인증키 (WFS/WMS·2D API·지오코더) + GIS건물통합정보 ...,미신청,NaN


### 묶음별 수집 확인

파일 존재·LFS 포인터 여부·취득 메타데이터.

In [4]:
for key in ["rainfall", "river", "small_tables", "sgis", "geo"]:
    m = node_metrics(f"h00_collect_{key}")
    print(f"{key:14} {m['files']:>3}개 · {m['bytes']/1e6:7.1f} MB · LFS 포인터 {m['lfs_pointers']}")

rainfall         1개 ·    12.6 MB · LFS 포인터 0


river            1개 ·     0.9 MB · LFS 포인터 0


small_tables     2개 ·     0.0 MB · LFS 포인터 0
sgis            36개 ·   373.4 MB · LFS 포인터 0


geo             22개 ·   226.4 MB · LFS 포인터 0


### 원본 계약 검증 직접 실행

강수 묶음만 지금 검증해 본다. 파이프라인 노드 `h01_contract_rainfall` 이 부르는 함수와 같다.

In [5]:
from src.data.validate_raw import validate_contract
from src.stages.h01_contract import GROUPS

report = validate_contract(
    ROOT / "config/data_contracts.yaml",
    fail_on="warning",
    waiver_path=ROOT / "config/raw_quality_waivers.yaml",
    only=GROUPS["rainfall"],
)
print(report["status"], report["finding_counts"])
pd.DataFrame(report["datasets"][0]["findings"])[["severity", "code", "message", "waived"]]

pass {'info': 0, 'warning': 0, 'error': 0, 'waived': 5, 'waiver_errors': 0}


,severity,code,message,waived
0,warning,exact_duplicates,완전 중복 행이 허용치를 초과합니다.,True
1,warning,duplicate_keys,키 중복 행이 허용치를 초과합니다.,True
2,warning,invalid_dates,파싱할 수 없는 날짜가 허용치를 초과합니다.,True
3,warning,numeric_range,수치 범위를 벗어난 값이 있습니다.,True
4,warning,date_range,분석 허용 범위를 벗어난 날짜가 있습니다.,True


### 전체 검증 결과

구조 오류(error)는 0이어야 한다. 품질 문제(warning)는 사유·정제규칙·만료일을 적어
승인한 것만 H02 로 넘어간다.

In [6]:
rv = json.loads((ROOT / "artifacts/validation/raw_validation.json").read_text(encoding="utf-8"))
print(rv["status"], rv["finding_counts"])
pd.DataFrame([
    {
        "데이터": d["name"],
        "파일": len(d["files"]),
        "error": sum(1 for f in d["findings"] if f["severity"] == "error" and not f.get("waived")),
        "미승인 warning": sum(1 for f in d["findings"] if f["severity"] == "warning" and not f.get("waived")),
        "waiver 적용": sum(1 for f in d["findings"] if f.get("waived")),
    }
    for d in rv["datasets"]
])

pass {'info': 0, 'warning': 0, 'error': 0, 'waived': 8, 'waiver_errors': 0}


,데이터,파일,error,미승인 warning,waiver 적용
0,rainfall_hourly,1,0,0,5
1,river_level_hourly,1,0,0,3
2,pump_stations,1,0,0,0
3,river_metadata,1,0,0,0
4,sgis_aggregation_age,5,0,0,0
5,sgis_aggregation_summary,10,0,0,0
6,sgis_grid_statistics,12,0,0,0
7,sgis_aggregation_boundaries,25,0,0,0
8,sgis_grid_boundaries,20,0,0,0
9,land_cover_middle,64,0,0,0


### 승인된 waiver

각 항목을 H02 에서 어떤 규칙으로 처리할지가 정해져 있다.

In [7]:
import yaml

waivers = yaml.safe_load((ROOT / "config/raw_quality_waivers.yaml").read_text(encoding="utf-8"))["waivers"]
pd.DataFrame(waivers)[["dataset", "code", "cleaning_rule", "expires_on"]]

,dataset,code,cleaning_rule,expires_on
0,rainfall_hourly,exact_duplicates,"H02-R01 완전중복 quarantine 후 1건 유지, 행수 보존식 제출",2026-08-22
1,rainfall_hourly,duplicate_keys,"H02-R02 충돌행 분리, 값이 다르면 자동 선택 금지",2026-08-22
2,rainfall_hourly,invalid_dates,H02-R03 잘못된 날짜 3행 quarantine,2026-08-22
3,rainfall_hourly,numeric_range,"H02-R04 원값·원행 보존, 코드북 확인 전 이벤트 분석 제외",2026-08-22
4,rainfall_hourly,date_range,"H02-R05 2000년 이전 9행 quarantine, 2015~2024 고정기간 사용",2026-08-22
5,river_level_hourly,sentinel_values,"H02-W01 코드북 확인 후 결측 전환, 원값·플래그 보존",2026-08-22
6,river_level_hourly,numeric_range,"H02-W02 -100~1000 밖 18셀 quarantine, 원값·원행 보존 후...",2026-08-22
7,river_level_hourly,temporal_coverage,H02-W03 수위를 도시 전체 공간검증·음성라벨에서 제외하고 특정 호우 보조 사례...,2026-08-22


### 다음

P1 검증 노드는 R1 승인이 있어야 P2 가 열린다.

```bash
python -m src.pipeline approve h01_contract_rainfall --by R1 --note "waiver 사유 확인"
```